# Wrap a live game as an MCP server

By the end of this notebook Claude will be building bridges in the **pygame window
beside you**, in real time, and choosing them the way a client asks.

**Layout (keep all three visible):**

1. This notebook
2. The Bridge Builder window
3. Claude Code

The server grows in three stages. Each stage adds one kind of MCP building block,
and each time we ask Claude **the same thing**: *"Build me the cheapest bridge."*
Watch how the answer changes.

| Stage | Adds | File | What the model can do |
|---|---|---|---|
| 1 | **Tools** | `mcp_tools.py` | act: look, build, test |
| 2 | **Resources** (+ a prompt) | `mcp_designs.py` | know: level facts, tested designs |
| 3 | **Prompts** carrying a client brief | `mcp_clients.py` | follow a client's requirements |

## 0. Start the game at stage 1

In a terminal, from the repo root (already `uv sync`'d):

```bash
uv run python -m bridge_builder --stage 1
```

Red **anchor** circles on both banks. Click an anchor, drag to a neighbour.
**Space** sends the train. **R** resets. Leave the window running.

To move to the next stage later: close the window, start it again with the next
`--stage`, then run `/mcp` in Claude Code to reconnect. Fell behind? `--stage 3`
is the finished kit.

## Stage 1: tools, so the model can act

A tool is an ordinary Python function plus `@mcp.tool()`. The SDK publishes the
function name, its docstring and its type hints as the tool the model sees.

In [ ]:
from pathlib import Path
import bridge_builder.mcp_tools as mcp_tools

print(Path(mcp_tools.__file__).read_text(encoding="utf-8"))

### Ping the game and register it with Claude

`register_claude()` writes `.mcp.json` in this project (and the Claude Desktop
config if that app is installed). Claude Code picks up project MCP servers from
that file.

In [ ]:
from bridge_builder.client import (
    call_tool,
    get_prompt,
    list_prompts,
    list_resources,
    list_tools,
    read_resource,
    register_claude,
)

print("tools:", list_tools())
print(call_tool("status"))
register_claude()

If Claude Code is already open on this folder, run `/mcp` (or reconnect) **once**.
`bridge-engineer` should then be listed as an HTTP server at
`http://127.0.0.1:8765/mcp`.

### Drive the game with the same tools the model will use

Build a **naive straight deck** at `y=3`, then send the train.

In [ ]:
call_tool("reset")
for x in range(-6, 6, 2):
    print(x, "->", x + 2, call_tool("place_girder", x1=x, y1=3, x2=x + 2, y2=3))
call_tool("status")

In [ ]:
call_tool("start_train")

Watch the window. A straight line of pinned beams is a chain: it sags, the beams
glow red, and one snaps. A line is not a bridge.

In [ ]:
call_tool("reset")

### Ask Claude

In Claude Code:

> Build me the cheapest bridge.

It has tools but no knowledge of this level: it improvises. It may build a truss,
it may forget a diagonal and watch the bridge fold. Note what it built and what it
cost.

## Stage 2: resources, so the model can know

Restart the game with `--stage 2`, then `/mcp` in Claude Code.

A resource is read-only content the model can pull into context, published with
`@mcp.resource(uri)`. This stage publishes the level's facts and four tested
reference designs (cards and exact beam lists from `knowledge/designs/`), plus the
`design` prompt that builds one of them.

In [ ]:
import bridge_builder.mcp_designs as mcp_designs

print(Path(mcp_designs.__file__).read_text(encoding="utf-8"))

In [ ]:
print("resources:", list_resources())
print("prompts:", list_prompts())
print(read_resource("bridge://designs"))

### Ask Claude the same thing

> Build me the cheapest bridge.

The server instructions now tell the model to offer a choice first: a reference
design or a custom one. Ask for the cheapest and it builds the **inverted truss**
(2100). It ties with the overhead truss on cost but has far more margin, and the
catalog lists it first.

The prompt is also a slash command: `/bridge-engineer:design inverted_truss`.

## Stage 3: prompts carry a client's knowledge

Restart the game with `--stage 3`, then `/mcp` in Claude Code.

The tools and the designs know nothing about **who** the bridge is for. FETNIS has
requirements: ships must pass under the bridge, and it should be as sustainable as
possible. That knowledge lives in a brief, a Markdown file in `knowledge/clients/`,
published as a resource. The `brief` prompt puts the brief and the design catalog
in front of the model with one job: build what the client's requirements rank highest.

In [ ]:
import bridge_builder.mcp_clients as mcp_clients

print(Path(mcp_clients.__file__).read_text(encoding="utf-8"))

In [ ]:
print(read_resource("bridge://clients/fetnis"))

This is exactly what the model receives when you run the prompt:

In [ ]:
print(get_prompt("brief", client="fetnis"))

### Ask Claude, as FETNIS

> /bridge-engineer:brief fetnis

The cheapest design, the inverted truss, hangs below the road: ships cannot pass,
so the brief rules it out. Of the rest, the **timber arch** has by far the highest
wood share (18 of 28 beams). It costs 3160, over 1000 more than the cheapest, and
the model should tell you so.

**Where does knowledge belong?**

| Put it in | Reaches the model | Good for |
|---|---|---|
| Server instructions | every conversation | rules of the game |
| Resources | when the model (or you) reads them | facts, designs, briefs |
| Prompts | when **you** invoke them | a workflow, one client's requirements |

A prompt is a policy the model follows, not a rule the server enforces. If
clearance were a legal requirement, where would you put it so it *cannot* be broken?

## Exercise: write your own brief

Briefs are re-read on every request, so a new file works **without a restart**.
Run the cell, edit the file it prints, then in Claude Code:

> /bridge-engineer:brief harbour

Ideas: a harbour authority that only needs clearance and the lowest price; a city
that bans titanium; a low-profile bridge with nothing above `y=5`.

In [ ]:
from bridge_builder.clients import CLIENTS_DIR

brief = CLIENTS_DIR / "harbour.md"
brief.write_text("""# Harbour Authority

The cheapest bridge that ships can sail under.

## Requirements, in priority order

1. **Clearance (must).** No beam may hang below the deck: every beam inside the
   gap (x between -6 and 6) must have both ends at y >= 3.
2. **Cost.** The cheapest design that meets 1.
3. **Safety.** Peak beam load below 0.9.

## Report

The chosen design, its cost and peak load, and why the cheapest design overall
was or was not chosen.
""", encoding="utf-8")
print(brief)
print(read_resource("bridge://clients"))

## Reference: materials

`place_girder` takes an optional `material`. The level resource lists the table:

| material | cost | break force (pull) | stiffness | density |
|---|---|---|---|---|
| steel | 100 | 1.0x (950 N) | 1.0 | 1.0 |
| wood | 120 | 0.6x (570 N) | 0.5 | 0.5 |
| titanium | 400 | 2.0x (1900 N) | 0.8 | 0.8 |

Steel is the cheap default. Wood costs more and is weaker: you choose it for
sustainability. A beam breaks when its pull or push is too much; long diagonals
buckle under push at half the load of a 2 m beam. Joints turn freely, so only
triangles are rigid.

The grid also continues **below** the deck inside the gap (`x=-4..4`, `y=1` and
`y=-1`), so an inverted truss hanging under the road is a valid design. The bank
walls at `x=±6` are solid below the anchors.

## Spoiler

Working trusses (hide until someone is stuck). Importing them does **not** build
them: Claude still has to place each beam in order from an existing node.

In [ ]:
from bridge_builder.truss import INVERTED_TRUSS, TRUSS_GIRDERS

print("overhead truss (steel, 2100):")
for row in TRUSS_GIRDERS:
    print(row)

print("\ninverted, under the deck (steel, 2100):")
for row in INVERTED_TRUSS:
    print(row)